In [15]:
import pandas as pd
from io import StringIO

def expand_table_with_missing_bpm(df):
    # Skip the first two rows (header and units)
    data = df.iloc[1:].copy()
    
    # Convert columns to numeric where applicable
    data['HR'] = pd.to_numeric(data['HR'], errors='coerce')
    data['Calories'] = pd.to_numeric(data['Calories'], errors='coerce')
    
    # Create a list to store expanded rows
    expanded_rows = []
    
    # Iterate through rows to interpolate missing BPM values
    for i in range(len(data) - 1):
        current_row = data.iloc[i]
        next_row = data.iloc[i + 1]
        
        current_bpm = current_row['HR']
        next_bpm = next_row['HR']
        
        # Add the current row to the expanded rows
        expanded_rows.append(current_row)
        
        # Check if there are missing BPM values
        if next_bpm - current_bpm > 1:
            missing_bpm_count = int(next_bpm - current_bpm - 1)
            calorie_diff = (next_row['Calories'] - current_row['Calories']) / (missing_bpm_count + 1)
            
            # Generate missing rows
            for j in range(1, missing_bpm_count + 1):
                interpolated_bpm = current_bpm + j
                interpolated_calories = current_row['Calories'] + calorie_diff * j
                
                # Create a new row with interpolated values
                interpolated_row = current_row.copy()
                interpolated_row['HR'] = interpolated_bpm
                interpolated_row['Calories'] = interpolated_calories
                
                # Add the interpolated row to the expanded rows
                expanded_rows.append(interpolated_row)
    
    # Add the last row to the expanded rows
    expanded_rows.append(data.iloc[-1])
    
    # Convert the list of rows back to a DataFrame
    expanded_data = pd.DataFrame(expanded_rows)
    
    return expanded_data

# Example usage
csv_data = """Time	HR	VO2	VO2	VE/VO2	VCO2	VE/VCO2	RER	Calories	Fat	CHO
min:sec	BPM	mL/min	mL/kg/min		mL/kg/min			Cals/min	%	%
00:00	96	478	5.94	33.46	5.08	39.18	0.856	2.33	47.4	52.6
00:14	106	788	9.79	31.34	8.83	34.72	0.902	3.88	31.7	68.3"""

# Read the CSV data into a DataFrame
df = pd.read_csv(StringIO(csv_data), sep="\t")

# Keep only HR and Calories columns
df = df[['HR', 'Calories']]

# Expand the table
expanded_df = expand_table_with_missing_bpm(df)

# Display the expanded DataFrame
print(expanded_df)

      HR  Calories
1   96.0     2.330
1   97.0     2.485
1   98.0     2.640
1   99.0     2.795
1  100.0     2.950
1  101.0     3.105
1  102.0     3.260
1  103.0     3.415
1  104.0     3.570
1  105.0     3.725
2  106.0     3.880
